# Earth Science Exercise: Thermal Conductivity & Heat Flow
## Northeast German Basin (NEGB) Geothermal Aquifers

These exercises are based on real laboratory and borehole data published in:

> Fuchs, S., Förster, A. (2010). *Rock thermal conductivity of Mesozoic geothermal aquifers in the Northeast German Basin.* Chemie der Erde S3, 13-22.

The study measured thermal conductivity (λ) and porosity on sandstone core samples from eight
(geothermal) exploration boreholes, and combined these with high-precision borehole temperature
logs to estimate subsurface heat flow.

**Notebook structure**

- **Part A — Geothermics fundamentals**: Fourier's law, the geometric-mean
 model for calculating matrix thermal conductivity, and heat-flow calculation from
  temperature-gradient logs.
- **Part B — Machine Learning**: using the same dataset to build predictive and
  exploratory models — linear regression, a decision tree, a support vector machine,
  k-means clustering, and a neural network.

Cells marked **`# YOUR CODE HERE`** are for you to complete. Markdown cells with
**"Your answer:"** ask you to interpret results in your own words.


## Background

**Fourier's law of heat conduction** relates the conductive heat flow *q* to the
thermal conductivity λ and the temperature gradient dT/dz in a rock layer:

$$ q = -\lambda \frac{dT}{dz} $$

In borehole studies, λ is usually reported in W/(m·K), the temperature gradient
in °C/km, and heat flow in mW/m². Watch your unit conversions carefully!

**Geometric-mean model.** Because the sandstones are porous and water-saturated,
the measured bulk thermal conductivity (λ_bulk) is a mixture of the solid mineral
matrix (λ_matrix) and the pore-filling water (λ_water ≈ 0.6 W/(m·K)). The paper uses a
geometric-mean (volume-weighted) mixing law:

$$ \lambda_{bulk} = \lambda_{matrix}^{(1-\phi)} \cdot \lambda_{water}^{\phi} $$

where φ is the fractional porosity (0–1). Rearranged to solve for the matrix
conductivity from a bulk measurement:

$$ \lambda_{matrix} = \frac{\lambda_{bulk}}{\lambda_{water}^{\phi/(1-\phi)}}^{\,1/(1-\phi)}
\;\;\Longleftrightarrow\;\;
\log\lambda_{matrix} = \frac{\log\lambda_{bulk} - \phi\log\lambda_{water}}{1-\phi}$$

**Formation abbreviations used in the dataset** (from oldest/deepest to youngest/shallowest):

| Code | Formation | Age |
|---|---|---|
| smD | Detfurth | Middle Buntsandstein (Triassic) |
| smH | Hardegsen | Middle Buntsandstein (Triassic) |
| smS | Solling | Upper Buntsandstein (Triassic) |
| kmS | Stuttgart Fm. | Middle Keuper (Triassic) |
| kOPS | Postera | Rhaetian (Triassic) |
| kCs | Contorta | Rhaetian (Triassic) |
| juhe | Hettangian | Lower Jurassic (Lias) |
| jusi | Sinemurian | Lower Jurassic (Lias) |
| jupl | Pliensbachian | Lower Jurassic (Lias) |


## The dataset

The table below reproduces 74 individual core-sample measurements (Table 1 of the
paper): thermal conductivity of water-saturated samples measured with the optical
scanning method, corrected for in-situ temperature, together with the derived matrix
thermal conductivity and the effective (Archimedes) porosity, for eight boreholes in
three areas of the basin (Stralsund, Neubrandenburg, Schwerin).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Table 1 data: sandstone core samples (Fuchs & Foerster, 2010)
# Formation, Borehole, Depth [m], bulk lambda as-measured [W/m/K],
# bulk lambda corrected for in-situ T [W/m/K], matrix lambda mean [W/m/K], porosity [%]
_rows = [
("jupl","Dp N 1/82",991.2,3.6,3.5,5.5,21.4),
("jupl","Dp N 1/82",1017.0,3.1,3.0,6.1,26.1),
("jusi","Dp N 1/82",1134.6,3.2,3.1,5.6,28.4),
("jusi","Dp N 1/82",1136.0,3.2,3.0,5.6,28.2),
("juhe","Gt N 3/86",1120.5,3.3,3.2,5.2,22.4),
("juhe","Gt N 3/86",1122.6,3.6,3.5,6.7,24.8),
("juhe","Gt N 3/86",1124.3,3.5,3.3,5.1,21.1),
("juhe","Gt N 3/86",1125.7,4.0,3.8,4.5,16.5),
("juhe","Gt N 3/86",1144.2,3.0,2.9,7.6,32.4),
("juhe","Gt N 3/86",1145.9,3.4,3.3,6.2,26.9),
("juhe","Gt N 3/86",1150.7,3.2,3.1,6.1,27.4),
("juhe","Gt N 3/86",1153.1,3.3,3.2,7.4,31.5),
("juhe","Gt N 3/86",1154.0,3.2,3.1,5.5,25.7),
("juhe","Gt N 3/86",1157.5,3.3,3.1,5.5,26.7),
("juhe","Gt N 3/86",1159.3,3.3,3.2,6.9,29.6),
("kCs","Gt N 2/85",1222.1,3.4,3.2,5.2,20.0),
("kCs","Gt N 2/85",1225.3,3.2,3.1,6.3,25.3),
("kCs","Gt N 2/85",1229.4,3.8,3.6,5.5,18.7),
("kCs","Dp N 1/82",1252.0,3.5,3.3,4.7,21.9),
("kCs","Gt S 5/87",2063.2,4.0,3.7,6.8,25.5),
("kCs","Gt S 5/87",2072.1,4.2,3.8,6.7,22.8),
("kCs","Gt S 5/87",2072.7,4.1,3.7,7.4,23.7),
("kCs","Gt S 5/87",2072.9,4.5,4.0,6.1,20.0),
("kCs","Gt S 5/87",2109.5,4.4,4.0,6.5,20.1),
("kCs","Gt S 5/87",2110.5,4.2,3.8,6.1,20.5),
("kCs","Gt S 5/87",2112.4,3.7,3.4,6.2,22.3),
("kCs","Gt S 5/87",2113.1,4.0,3.6,5.5,18.8),
("kCs","Gt S 5/87",2114.2,4.2,3.8,7.2,23.0),
("kCs","Gt S 5/87",2115.2,4.1,3.7,8.0,22.6),
("kOPS","Gt S 5/87",2136.5,4.5,4.1,7.7,22.0),
("kOPS","Gt S 5/87",2136.9,4.1,3.7,7.1,22.1),
("kOPS","Dp N 1/82",1274.6,3.8,3.6,5.2,22.4),
("kOPS","Dp N 1/82",1275.0,3.7,3.5,6.3,26.3),
("kOPS","Dp N 1/82",1281.8,3.3,3.2,5.5,25.2),
("kOPS","Gt N 2/85",1255.5,3.6,3.5,7.8,30.5),
("kOPS","Gt N 2/85",1261.0,3.1,3.0,7.1,30.0),
("kmS","Gt N 2/85",1517.5,1.9,1.9,2.4,11.0),
("kmS","Gt N 2/85",1525.4,2.1,2.1,2.7,13.7),
("kmS","Gt N 2/85",1528.0,2.0,2.0,3.2,17.0),
("kmS","Gt N 2/85",1537.7,2.1,2.1,4.3,26.3),
("kmS","Gt N 2/85",1541.7,2.2,2.2,4.2,25.8),
("smS","Gt Ss 1/85",1404.6,2.6,2.5,4.1,19.0),
("smS","Gt Ss 1/85",1406.6,3.2,3.1,4.5,18.8),
("smS","Gt Ss 1/85",1408.2,3.2,3.1,5.3,23.2),
("smS","Gt Ss 1/85",1412.3,4.2,3.9,5.3,18.5),
("smS","Gt Ss 2/85",1448.1,3.4,3.3,5.3,19.8),
("smS","Gt Ss 2/85",1452.3,3.9,3.7,5.6,21.5),
("smS","Gt Ss 2/85",1454.3,3.2,3.0,4.5,16.9),
("smS","Gt Ss 2/85",1463.0,4.3,4.0,3.9,6.0),
("smH","Gt Ss 1/85",1424.0,2.8,2.7,4.7,24.0),
("smH","Gt Ss 1/85",1426.0,2.6,2.5,2.8,22.0),
("smH","Gt Ss 1/85",1430.4,2.8,2.7,4.4,25.0),
("smH","Gt Ss 1/85",1434.7,2.7,2.7,4.5,22.0),
("smH","Gt Ss 1/85",1435.6,2.9,2.8,4.4,24.0),
("smH","Gt Ss 2/85",1485.5,3.3,3.1,5.5,23.5),
("smH","Gt Ss 2/85",1489.5,3.6,3.5,6.2,24.1),
("smH","Gt Ss 2/85",1496.2,3.4,3.3,5.0,21.0),
("smH","Gt Ss 2/85",1504.9,3.1,2.9,4.6,23.5),
("smH","Gt Ss 2/85",1514.1,3.6,3.4,5.7,21.7),
("smH","Gt Ss 2/85",1518.7,3.3,3.2,6.5,26.4),
("smH","Gt Ss 2/85",1519.3,3.5,3.3,5.3,21.4),
("smD","Gt Ss 1/85",1467.4,3.8,3.6,4.8,19.1),
("smD","Gt Ss 1/85",1491.2,3.5,3.3,4.9,22.5),
("smD","Gt Ss 1/85",1530.3,3.0,2.9,4.6,19.1),
("smD","Gt Ss 1/85",1540.9,3.1,3.0,4.5,18.0),
("smD","Gt Ss 2/85",1533.9,3.7,3.5,5.1,17.1),
("smD","Gt Ss 2/85",1540.6,3.3,3.1,5.0,21.7),
("smD","Gt Ss 2/85",1545.2,3.0,2.9,4.2,21.0),
("smD","Gt Ss 2/85",1547.6,3.2,3.0,5.0,23.0),
("smD","Gt Ss 2/85",1560.1,3.5,3.3,5.5,21.6),
("smD","Gt Ss 2/85",1562.2,3.3,3.1,4.8,20.4),
("smD","Gt Ss 2/85",1568.9,3.8,3.6,6.6,23.7),
("smD","Gt Ss 2/85",1577.6,3.6,3.4,6.4,25.7),
("smD","Gt Ss 2/85",1602.1,3.5,3.4,3.8,9.6),
]

df = pd.DataFrame(_rows, columns=[
    "Formation", "Borehole", "Depth_m", "Lambda_bulk_meas",
    "Lambda_bulk_corr", "Lambda_matrix_mean", "Porosity_pct"
])

print(f"{len(df)} samples loaded")
df.head()


## Interval heat-flow data (Table 3)

These are the four/two homogeneous-gradient intervals identified in the Middle
Buntsandstein section of the two Stralsund boreholes, each with an average
temperature gradient and an average (in-situ corrected) thermal conductivity, used
by the authors to compute an interval heat flow.


In [ ]:
heat_flow_data = pd.DataFrame([
    # Borehole, Interval, Formation, Depth_top, Depth_bottom, Gradient [C/km], Lambda [W/m/K], Reported_q [mW/m2]
    ("Gt Ss 1/85", "I",   "smS",       1405.90, 1415.95, 23.5, 3.37, 79.3),
    ("Gt Ss 1/85", "II",  "smH",       1421.30, 1434.30, 27.3, 2.69, 73.3),
    ("Gt Ss 1/85", "III", "smH+smD",   1434.00, 1475.30, 22.7, 3.02, 68.4),
    ("Gt Ss 1/85", "IV",  "smD",       1483.80, 1498.10, 23.1, 3.29, 75.9),
    ("Gt Ss 2/85", "I",   "smS",       1446.70, 1456.40, 23.3, 3.52, 81.9),
    ("Gt Ss 2/85", "II",  "smH",       1484.85, 1521.10, 23.2, 3.24, 75.2),
], columns=["Borehole","Interval","Formation","Depth_top_m","Depth_bottom_m",
            "Gradient_C_per_km","Lambda_Wmk","Reported_q_mWm2"])
heat_flow_data


## Formation thermal-conductivity profile (Table 4, Gt Ss 1/85 borehole)

Once an average heat flow is established for a borehole (using the core-controlled
Buntsandstein interval), it can be combined with the *entire* temperature-gradient
log to indirectly estimate λ for formations that were never cored — including
Jurassic and Cretaceous claystones, limestones and marls above the sandstone
aquifers.


In [ ]:
profile_data = pd.DataFrame([
    # Depth[m], Formation code, Formation name, Gradient[C/km], Reported_lambda[W/m/K]
    (223,  "krt",   "Turonian",                       26.3, 2.8),
    (250,  "krc",   "Cenomanian",                      26.6, 2.8),
    (261,  "krl",   "Albian",                          24.7, 3.0),
    (282,  "krh",   "Hauterivian",                     28.7, 2.6),
    (356,  "jutc",  "Toarcian",                        50.8, 1.5),
    (460,  "juplo", "Domerian (Up. Pliensbachian)",    33.8, 2.2),
    (481,  "juplu", "Carixian (Low. Pliensbachian)",   28.9, 2.6),
    (666,  "jusiu+juhe", "Low. Sinemurian + Hettangian",25.7, 2.9),
    (690,  "kTs",   "Triletes",                        30.4, 2.4),
    (711,  "kCs",   "Contorta",                        28.3, 2.6),
    (753,  "kOPS",  "Upper Postera",                   29.2, 2.5),
    (783,  "kmSM2-3","Lower Postera",                  33.7, 2.2),
    (800,  "kmSM1", "Basisdolomit",                    26.0, 2.9),
    (819,  "kmS",   "Stuttgart Fm.",                   36.1, 2.1),
    (949,  "kmGu",  "Lower Gipskeuper",                40.0, 1.9),
    (1015, "ku",    "Lettenkeuper",                    36.9, 2.0),
    (1093, "mm",    "Hauptmuschelkalk",                41.3, 1.8),
    (1173, "mmAN",  "Anhydrit",                        35.6, 2.1),
    (1258, "mu",    "Wellenkalk",                      35.2, 2.1),
    (1275, "soMY",  "Myophorien",                      40.4, 1.8),
    (1374, "soPR",  "Pelitroet",                       37.4, 2.0),
    (1393, "soSR",  "Salinarroet",                     34.1, 2.2),
    (1421, "smS",   "Solling",                         30.6, 2.4),
    (1463, "smH",   "Hardegsen",                       24.2, 3.1),
    (1510, "smDW",  "Detfurth (alt. sequence)",        24.4, 3.0),
], columns=["Depth_m","Code","Formation_name","Gradient_C_per_km","Reported_lambda_Wmk"])
profile_data


---
# Part A — Geothermics Fundamentals


### A1. Explore the dataset

Group the sample data by `Formation` and compute the **mean** and **standard
deviation** of `Lambda_bulk_corr` and `Porosity_pct` for each formation.


In [5]:
summary = pd.DataFrame(columns=[
    "Formation", "Lambda_bulk_corr_mean", "Lambda_bulk_corr_std",
    "Porosity_pct_mean", "Porosity_pct_std"
])

for formation in df["Formation"].unique():
    form = df[df["Formation"] == formation]

    lambda_mean = form["Lambda_bulk_corr"].mean()
    lambda_std = form["Lambda_bulk_corr"].std()
    porosity_mean = form["Porosity_pct"].mean()
    porosity_std = form["Porosity_pct"].std()

    summary.loc[len(summary)] = [formation, lambda_mean, lambda_std, porosity_mean, porosity_std]

summary

,Formation,Lambda_bulk_corr_mean,Lambda_bulk_corr_std,Porosity_pct_mean,Porosity_pct_std
0,jupl,3.250000,0.353553,23.750000,3.323402
1,jusi,3.050000,0.070711,28.300000,0.141421
2,juhe,3.245455,0.238175,25.909091,4.648538
3,kCs,3.621429,0.277845,21.800000,2.193697
4,kOPS,3.514286,0.353217,25.500000,3.637765
5,kmS,2.060000,0.114018,18.760000,6.988061
6,smS,3.325000,0.509201,17.962500,5.204651
7,smH,3.008333,0.331548,23.216667,1.622475
8,smD,3.238462,0.253438,20.192308,3.976485


**Your answer:** Compare your computed means to the bold (formation-average) values
quoted in Section 4.1 of the paper (e.g. Stuttgart Fm. ≈ 2.1 W/m/K, Postera ≈ 3.4–3.9
W/m/K, Middle Buntsandstein ≈ 2.7–3.5 W/m/K). Do they agree?


### A2. Porosity vs. bulk thermal conductivity

Make a scatter plot of `Porosity_pct` (x-axis) against `Lambda_bulk_corr` (y-axis),
with points colored by `Formation`.


In [ ]:
# TODO: create a scatter plot, colored by Formation, with a legend and axis labels
# YOUR CODE HERE


**Your answer:** Describe the trend you see. Higher porosity should *lower* the
bulk thermal conductivity of a water-saturated sandstone, because quartz
(λ ≈ 7.7 W/m/K) conducts heat far better than water (λ ≈ 0.6 W/m/K) — but with all
formations pooled together, is that trend actually clear in the scatter plot, or
does it look surprisingly weak/scattered?


### A3. The geometric-mean model

Implement the geometric-mean model to back out matrix thermal conductivity
(`Lambda_matrix_calc`) from the bulk, in-situ-corrected conductivity and the
porosity, using λ_water = 0.6 W/(m·K):

$$\log\lambda_{matrix} = \frac{\log\lambda_{bulk} - \phi\log\lambda_{water}}{1-\phi}$$

Then compare `Lambda_matrix_calc` to the paper's reported `Lambda_matrix_mean`
(which the authors computed differently — as the *average of separate dry and
saturated measurements* — so don't expect a perfect match).


In [ ]:
def lambda_matrix_from_bulk(lambda_bulk, porosity_pct, lambda_water=0.6):
    '''Return matrix thermal conductivity via the geometric-mean model.
    porosity_pct is in percent (0-100); internally convert to a fraction phi.
    '''
    phi = None  # YOUR CODE HERE: convert porosity_pct to a 0-1 fraction
    log_lambda_matrix = None  # YOUR CODE HERE: apply the rearranged formula
    return None  # YOUR CODE HERE: return 10**log_lambda_matrix

df["Lambda_matrix_calc"] = lambda_matrix_from_bulk(df["Lambda_bulk_corr"], df["Porosity_pct"])

# TODO: compute the mean absolute error between Lambda_matrix_calc and Lambda_matrix_mean
mae = None  # YOUR CODE HERE
print(f"Mean absolute error: {mae:.2f} W/m/K")

# TODO: make a scatter plot of Lambda_matrix_mean (x) vs Lambda_matrix_calc (y) with a 1:1 line
# YOUR CODE HERE


**Your answer:** Are the two matrix-conductivity estimates close? What might
explain systematic differences between a value calculated with a simple mixing law
and one measured directly on dry and saturated samples (think about mineralogy,
pore geometry/cementation, and the assumptions built into the geometric-mean law)?


### A4. Computing heat flow from Fourier's law

Using the `heat_flow_data` table above, compute the heat flow for each interval
with Fourier's law:

$$ q \;[\text{mW/m}^2] = \lambda\;[\text{W/(m·K)}] \times \text{gradient}\;[^\circ\text{C/km}] $$

(Note: 1 W/(m·K) × 1 °C/km = 1 mW/m² — the km-to-m unit conversion and the
W-to-mW conversion cancel out, which is a handy coincidence worth verifying for
yourself.)

Compare your computed values to the `Reported_q_mWm2` column and compute the
percent error.


In [ ]:
# TODO: compute heat flow q = lambda * gradient for every row, store as a new column "Computed_q_mWm2"
heat_flow_data["Computed_q_mWm2"] = None  # YOUR CODE HERE

# TODO: compute the percent error vs. the reported value
heat_flow_data["Percent_error"] = None  # YOUR CODE HERE

heat_flow_data


**Your answer:** How large is the agreement between your Fourier's-law
calculation and the paper's reported interval heat flow? The paper reports average
heat flow of 74.2 ± 4.6 mW/m² (Gt Ss 1/85) and 78.5 ± 4.8 mW/m² (Gt Ss 2/85) — do
your per-interval values scatter around these averages in a way consistent with
the quoted uncertainty?


---
# Part B — Machine Learning Exercise

We now treat the 74-sample dataset and
apply standard machine-learning techniques: **linear regression**, a **decision
tree**, a **support vector machine**, **k-means clustering**, and a
**neural network**.

> Note: 74 samples is a *small* dataset by ML standards. Treat any result with
> appropriate caution, and think about what a larger, basin-wide database (which
> the authors mention as future work) would add.


### B0. Imports and train/test split

We will repeatedly need features `X = [Porosity_pct, Depth_m]` (and sometimes
`Lambda_bulk_corr`) and either a continuous target (`Lambda_bulk_corr` or
`Lambda_matrix_mean`) or a categorical target (`Formation`).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, silhouette_score
)
from sklearn.neural_network import MLPRegressor

RANDOM_STATE = 42


### B1. Linear regression — predicting bulk thermal conductivity from porosity

Fit a simple linear regression of `Lambda_bulk_corr` on `Porosity_pct`
(train/test split 70/30). Report the slope, intercept, R², and RMSE on the test
set. Then fit a **multiple** linear regression adding `Depth_m` as a second
feature and see whether it improves the fit.


In [ ]:
# --- Simple linear regression: Lambda_bulk_corr ~ Porosity_pct ---
X_simple = df[["Porosity_pct"]].values
y = df["Lambda_bulk_corr"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.3, random_state=RANDOM_STATE
)

# TODO: create and fit a LinearRegression model
lr_simple = None  # YOUR CODE HERE

# TODO: predict on the test set and compute R2 and RMSE
y_pred = None  # YOUR CODE HERE
r2 = None       # YOUR CODE HERE
rmse = None     # YOUR CODE HERE

print(f"Slope: {lr_simple.coef_[0]:.4f}   Intercept: {lr_simple.intercept_:.4f}")
print(f"Test R2: {r2:.3f}   Test RMSE: {rmse:.3f} W/m/K")

# --- Multiple linear regression: Lambda_bulk_corr ~ Porosity_pct + Depth_m ---
# TODO: repeat the above using X = df[["Porosity_pct", "Depth_m"]]
# YOUR CODE HERE


**Your answer:** Is the slope of the simple regression negative or positive, and
how large is the test R²? How this connects directly to what you found in A2? Did adding `Depth_m` show any effect?

### B2. Decision tree — classifying the formation

Use `[Porosity_pct, Lambda_bulk_corr, Depth_m]` as features to predict
`Formation` with a `DecisionTreeClassifier` (train/test split 70/30, stratified
by formation). Report test accuracy, and plot the
learned tree (limit `max_depth` to keep it readable, e.g. 4).


In [ ]:
features = ["Porosity_pct", "Lambda_bulk_corr", "Depth_m"]
X = df[features].values

le = LabelEncoder()
y_labels = le.fit_transform(df["Formation"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_labels, test_size=0.3, random_state=RANDOM_STATE, stratify=y_labels
)

# TODO: create and fit a DecisionTreeClassifier (try max_depth=4)
tree_clf = None  # YOUR CODE HERE

# TODO: predict on the test set and compute accuracy
y_pred_tree = None  # YOUR CODE HERE
acc_tree = None      # YOUR CODE HERE
print(f"Decision tree test accuracy: {acc_tree:.2f}")

# TODO: plot the tree itself with plot_tree(...), using feature_names=features and class_names=le.classes_
# YOUR CODE HERE

# TODO: print tree_clf.feature_importances_ next to `features` -- which feature matters most?


**Your answer:** With only 74 samples spread across 9 formations, some classes
have very few test examples — how does that affect your confidence in the
accuracy number? Which feature ended up with the highest importance, and does
that make physical sense given what you found in Part A?


### B3. Support vector machine — same classification task

Repeat the formation classification with a `SVC` (support vector classifier).
Standardize the features first (SVMs are sensitive to feature scale!). Try both a
linear kernel and an RBF kernel and compare test accuracy to the decision tree.


In [ ]:
# TODO: standardize X_train / X_test using StandardScaler (fit on train, transform both)
scaler = StandardScaler()
X_train_scaled = None  # YOUR CODE HERE
X_test_scaled = None   # YOUR CODE HERE

# TODO: fit an SVC with kernel="linear" and one with kernel="rbf"; compare test accuracy for both
svc_linear = None  # YOUR CODE HERE
svc_rbf = None      # YOUR CODE HERE

acc_linear = None  # YOUR CODE HERE
acc_rbf = None      # YOUR CODE HERE

print(f"SVM (linear kernel) accuracy: {acc_linear:.2f}")
print(f"SVM (RBF kernel) accuracy:    {acc_rbf:.2f}")
print(f"Decision tree accuracy (B2):  {acc_tree:.2f}")


**Your answer:** Did standardizing the features matter here? Which kernel
performed better, and which model overall (tree vs. linear SVM vs. RBF SVM) would
you trust more for this dataset, and why?


### B4. K-means clustering — does an unsupervised method rediscover the formations?

Cluster the samples using only `Porosity_pct` and `Lambda_bulk_corr`
(standardized), *without* telling the algorithm the formation labels.

1. Run k-means for k = 2..8, plot the inertia (elbow method) and silhouette score
   to help choose k.
2. Fit k-means with your chosen k and cross-tabulate the resulting cluster labels
   against the true `Formation` column with `pd.crosstab`.
3. Visualize the clusters (color) on the porosity-vs-conductivity scatter plot
   from A2 and compare by eye to the formation-colored version.


In [ ]:
X_cluster = df[["Porosity_pct", "Lambda_bulk_corr"]].values
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

# TODO: loop over k = 2..8, fit KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10),
#       and record inertia_ and silhouette_score for each k
inertias = []
silhouettes = []
ks = range(2, 9)
for k in ks:
    km = None  # YOUR CODE HERE
    inertias.append(None)     # YOUR CODE HERE
    silhouettes.append(None)  # YOUR CODE HERE

fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].plot(list(ks), inertias, "o-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow method")
axes[1].plot(list(ks), silhouettes, "o-", color="orange")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score"); axes[1].set_title("Silhouette method")
plt.tight_layout()
plt.show()

# TODO: pick a value of k based on the plots above and fit the final KMeans model
k_final = None  # YOUR CODE HERE
kmeans_final = None  # YOUR CODE HERE
df["Cluster"] = None  # YOUR CODE HERE

# TODO: cross-tabulate Cluster vs. Formation
# YOUR CODE HERE

# TODO: scatter plot colored by Cluster (compare visually to the Formation-colored plot in A2)
# YOUR CODE HERE


**Your answer:** Do the k-means clusters line up reasonably well with real
formations? What does this tell you about the limits of using only
petrophysical properties — without stratigraphic/positional information — to
recover lithostratigraphy?


### B5. Neural network regression

As an extension, train a `MLPRegressor` to predict
`Lambda_matrix_mean` from `[Porosity_pct, Depth_m, Lambda_bulk_corr]`, and compare
its test RMSE to a linear regression baseline on the same features. Neural
networks need scaled inputs and may need more data than we have here to
truly shine — treat this as a demonstration rather than a definitive comparison.


In [ ]:
feat_nn = ["Porosity_pct", "Depth_m", "Lambda_bulk_corr"]
X_nn = df[feat_nn].values
y_nn = df["Lambda_matrix_mean"].values

X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_nn, y_nn, test_size=0.3, random_state=RANDOM_STATE
)

scaler_nn = StandardScaler()
X_train_nn_scaled = scaler_nn.fit_transform(X_train_nn)
X_test_nn_scaled = scaler_nn.transform(X_test_nn)

# TODO: baseline linear regression on the same scaled features
lr_baseline = None  # YOUR CODE HERE
rmse_lr = None       # YOUR CODE HERE

# TODO: MLPRegressor with e.g. hidden_layer_sizes=(16,8), max_iter=5000, random_state=RANDOM_STATE
mlp = None  # YOUR CODE HERE
rmse_mlp = None  # YOUR CODE HERE

print(f"Linear regression test RMSE: {rmse_lr:.3f} W/m/K")
print(f"Neural network test RMSE:    {rmse_mlp:.3f} W/m/K")


**Your answer:** Did the neural network actually beat the linear-regression
baseline here? What would you need (more samples? more features? regularization?) before trusting a neural network on a
dataset like this in a real geothermal-exploration workflow?
